In [ ]:
from geoscore_de.data_flow.features.municipality import MunicipalityFeature

municipalities = MunicipalityFeature("../../data/raw/municipalities_2022.csv")
municipalities = municipalities.load_transform()
municipalities

In [ ]:
from geoscore_de.data_flow.features.migration import MigrationFeature

migration = MigrationFeature("../../data/raw/features/12711-91-01-5-migration.csv").load_transform()

In [ ]:
migration = migration.merge(municipalities, on="AGS", how="right")
migration

In [ ]:
migration["prop_net_migration_2024"] = migration["net_migration_2024"] / migration["Persons"]

In [ ]:
import plotnine as gg

# show rate of in/out migration as histogram
(
    gg.ggplot(migration)
    + gg.aes(x="prop_net_migration_2024")
    + gg.geom_histogram()
    + gg.labs(title="In/Out Migration Ratio Distribution")
)

In [ ]:
# now show it on map of germany
from geoscore_de.data_flow.geo import load_geo_data

gdf = load_geo_data("../../data/georef-germany-gemeinde.csv")

In [ ]:
gdf_merged = gdf.merge(migration, left_on="AGS", right_on="AGS", how="left", indicator=True)

In [ ]:
# print count of missing values
gdf_merged["_merge"].value_counts()

In [ ]:
import matplotlib.pyplot as plt

# use colormap that center is 1 (in_out_ratio = 1 means equal in and out migration)
gdf_merged.plot(
    column="prop_net_migration_2024",
    legend=True,
    cmap="coolwarm",
    figsize=(10, 10),
    vmin=-0.1,
    vmax=0.1,
    missing_kwds={"color": "black"},
)
plt.title("Migration in German Municipalities 2024")
plt.show()

## Migration update

Lets compare migration update from 2023 with the one from 2024.

In [ ]:
gdf_merged["yty_update"] = (gdf_merged["net_migration_2024"] - gdf_merged["net_migration_2023"]) / gdf_merged["Persons"]
gdf_merged["yty_update"]

In [ ]:
import matplotlib.pyplot as plt

# use colormap that center is 1 (in_out_ratio = 1 means equal in and out migration)
gdf_merged.plot(
    column="yty_update",
    legend=True,
    cmap="coolwarm",
    figsize=(10, 10),
    vmin=-0.1,
    vmax=0.1,
    missing_kwds={"color": "black"},
)
plt.title("Year to Year update in migration in German Municipalities 2024")
plt.show()